# Insurance Claim Fraud Scoring

### Fraud Detection Aml Analytics — Banking-AI

This notebook explains each step in plain language so new learners can follow:

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), familiarity with `scikit-learn` basics, and basic understanding of fraud detection and anti-money laundering terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Explain the fraud detection and anti-money laundering prediction task and why regression modelling supports this banking decision.
- Implement a regression workflow and evaluate predictive accuracy using domain-appropriate error metrics.
- Evaluate whether the prediction error is acceptable for the operational decision it supports.

**Estimated time:** 35–45 minutes

**Collection context:** Fraud detection, AML compliance, anomaly detection, and financial crime analytics in banking.

## Step 1 — Banking Problem Introduction

**Why this notebook matters:** Insurance fraud costs the industry billions annually. By scoring claims as they come in, insurers can fast-track legitimate claims and flag suspicious ones for manual adjusters.

**Input data used:** Claim amount, age of policy, number of vehicles involved, previous claims, and "witness present" indicator.

**What we predict:** A probability or score (`fraud_score`) indicating the likelihood of fraud.

**ML method used:** Random Forest Regressor (to produce a continuous score).

**Learning goal:** Learn how to use regression to create a risk scoring system.

**Primary stakeholders:** fraud analysts, compliance officers, risk managers, and financial crime investigators.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment ready.")


## Step 3 — Data Creation & Context

Build Synthetic Insurance Claims

We generate 3,000 claims. We'll inject fraud indicators like "Claim very soon after policy start" and "High claim amount with no witnesses."

In [ ]:
n = 3000
rng = RNG

policy_age_days = rng.integers(1, 3650, n)
claim_amount = rng.gamma(shape=2, scale=2000, n)
total_vehicles = rng.choice([1, 2, 3], n, p=[0.7, 0.2, 0.1])
witnesses = rng.choice([0, 1, 2], n, p=[0.4, 0.4, 0.2])
past_claims = rng.poisson(0.5, n)

# Logic for Fraud Risk (The underlying "Truth" we want to score)
risk_score = (
    1.5 * (policy_age_days < 30) +        # Claimed immediately after buying
    1.2 * (claim_amount > 10000) +        # Expensive claim
    1.0 * (witnesses == 0) +              # No witnesses
    0.8 * (past_claims > 2) +             # Frequent claimant
    rng.normal(0, 0.5, n)
)

# Normalize risk to a 0-100 scale
risk_score = (risk_score - risk_score.min()) / (risk_score.max() - risk_score.min()) * 100

df = pd.DataFrame({
    'policy_age_days': policy_age_days,
    'claim_amount': claim_amount,
    'total_vehicles': total_vehicles,
    'witnesses': witnesses,
    'past_claims': past_claims,
    'fraud_score_target': risk_score
})

print("Sample of generated claims:")
print(df.head())

## Step 4 — Exploratory Data Analysis

For this workflow, primary exploration happens through the operational visualisations in later steps.

## Step 5 — Feature Engineering

Prepare features for modelling. Domain-meaningful transformations help the model learn patterns aligned with banking practice.

In [ ]:
# Features are used as created in Step 3.
print("Target column: 'fraud_score_target'")

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: always predict the training-set mean ---
baseline_reg = DummyRegressor(strategy='mean')
baseline_reg.fit(X_train, y_train)
baseline_pred_bl = baseline_reg.predict(X_test)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred_bl))
print(f"Baseline RMSE (predict-mean): {baseline_rmse:.4f}")
print("Any useful model must produce a lower RMSE.")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

We use a Random Forest Regressor to predict the risk score.

In [ ]:
X = df.drop('fraud_score_target', axis=1)
y = df['fraud_score_target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

model = RandomForestRegressor(n_estimators=100, random_state=RANDOM_SEED)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.3)
plt.plot([0, 100], [0, 100], '--r')
plt.xlabel('Actual Risk Score')
plt.ylabel('Predicted Fraud Score')
plt.title('Actual vs Predicted Fraud Score')
plt.tight_layout()
plt.show()
plt.close()

importances = pd.Series(model.feature_importances_, index=X.columns)
importances.sort_values().plot(kind='barh')
plt.title('Key Drivers of Insurance Fraud Score')
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Bar chart, scatter plot, line chart titled "Actual vs Predicted Fraud Score" and "Key Drivers of Insurance Fraud Score" with Actual Risk Score on the x-axis and Predicted Fraud Score on the y-axis. The chart highlights spatial separation or clustering patterns in the data.

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

**Summary:**
- Claims with **high amounts** and **no witnesses** that happen **shortly after a policy starts** receive the highest fraud scores.
- By setting a threshold (e.g., Score > 80), the insurance company can automatically flag the most suspicious claims for deep investigation.

**Learning Outcome:** You have learned how to build a continuous risk-scoring model that translates multiple behavioral "red flags" into a single actionable number.

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: inspect predictions at different points ---
print("Operational demonstration — model predictions across the test range:\n")
low_idx  = int(np.argmin(y_test.values))
high_idx = int(np.argmax(y_test.values))
mid_idx  = int(np.argsort(y_test.values)[len(y_test) // 2])

for label, idx in [("Low-end", low_idx), ("Mid-range", mid_idx), ("High-end", high_idx)]:
    actual = y_test.values[idx]
    pred   = y_pred[idx]
    print(f"  {label}  actual: {actual:.4f}  predicted: {pred:.4f}  error: {abs(actual-pred):.4f}")

print("\nPractitioners would use these predictions as one input among several.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end fraud detection and anti-money laundering workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- False positives can freeze legitimate accounts, causing financial hardship and customer distrust.
- Fraud models risk encoding biases against specific demographic groups or geographic regions.
- Transaction monitoring must balance fraud prevention with customer privacy rights.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in fraud detection and anti-money laundering?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.